In [ ]:
vgg_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

vgg_train_dataset = datasets.ImageFolder(train_dir, transform=vgg_transform)
vgg_val_dataset = datasets.ImageFolder(val_dir, transform=vgg_transform)
vgg_test_dataset = datasets.ImageFolder(test_dir, transform=vgg_transform)

vgg_batch_size = 16

vgg_train_loader = DataLoader(
    vgg_train_dataset,
    batch_size=vgg_batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

vgg_val_loader = DataLoader(
    vgg_val_dataset,
    batch_size=vgg_batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

vgg_test_loader = DataLoader(
    vgg_test_dataset,
    batch_size=vgg_batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print("Device:", device)
print("Train batches:", len(vgg_train_loader))
print("Val batches:", len(vgg_val_loader))
print("Test batches:", len(vgg_test_loader))

Device: cuda
Train batches: 389
Val batches: 109
Test batches: 50


In [ ]:
def build_vgg16_model(num_classes):
    weights = VGG16_Weights.DEFAULT
    model = vgg16(weights=weights)
    for param in model.features.parameters():
        param.requires_grad = False
    model.classifier = nn.Sequential(
        nn.Linear(512 * 7 * 7, 512),
        nn.ReLU(),
        nn.Dropout(p=0.5),
        nn.Linear(512, num_classes)
    )
    return model

In [ ]:
vgg_config = {
    "model_name": "vgg16_transfer_learning",
    "model_title": "VGG16 Transfer Learning",
    "architecture": "pretrained VGG16 with frozen convolutional features and compact classifier",
    "image_size": 224,
    "batch_size": vgg_batch_size,
    "num_epochs": 10,
    "learning_rate": 0.001,
    "optimizer": "Adam",
    "loss_function": "CrossEntropyLoss",
    "num_classes": num_classes,
    "pretrained": True,
    "pretrained_dataset": "ImageNet",
    "normalization": "ImageNet mean/std",
    "classifier": "Linear(25088, 512) + ReLU + Dropout(0.5) + Linear(512, 6)",
    "seed": 42
}

vgg_path = MODELS_DIR / "vgg16_transfer_learning_best.pth"

In [ ]:
vgg_model = build_vgg16_model(num_classes).to(device)
vgg_result = train_model(
    vgg_model,
    vgg_train_loader,
    vgg_val_loader,
    vgg_test_loader,
    vgg_train_dataset.classes,
    vgg_config,
    vgg_path
)

print("Best val accuracy:", round(vgg_result["best_val_accuracy"], 4))
print("Test accuracy:", round(vgg_result["test_accuracy"], 4))
print("Test macro F1:", round(vgg_result["test_macro_f1"], 4))
print("Test weighted F1:", round(vgg_result["test_weighted_f1"], 4))


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:07<00:00, 70.2MB/s]


Epoch [1/10] train_loss: 0.7126, train_acc: 0.7651, val_loss: 0.4137, val_acc: 0.8349
Epoch [2/10] train_loss: 0.3029, train_acc: 0.8920, val_loss: 0.4641, val_acc: 0.8544
Epoch [3/10] train_loss: 0.2291, train_acc: 0.9232, val_loss: 0.4291, val_acc: 0.8681
Epoch [4/10] train_loss: 0.1760, train_acc: 0.9404, val_loss: 0.5371, val_acc: 0.8681
Epoch [5/10] train_loss: 0.1246, train_acc: 0.9615, val_loss: 0.6160, val_acc: 0.8681
Epoch [6/10] train_loss: 0.1839, train_acc: 0.9488, val_loss: 0.6391, val_acc: 0.8681
Epoch [7/10] train_loss: 0.1616, train_acc: 0.9544, val_loss: 0.7292, val_acc: 0.8572
Epoch [8/10] train_loss: 0.1671, train_acc: 0.9551, val_loss: 0.6722, val_acc: 0.8612
Epoch [9/10] train_loss: 0.1504, train_acc: 0.9607, val_loss: 0.7829, val_acc: 0.8578
Epoch [10/10] train_loss: 0.1542, train_acc: 0.9623, val_loss: 0.8032, val_acc: 0.8693
Best val accuracy: 0.8693
Test accuracy: 0.8777
Test macro F1: 0.9079
Test weighted F1: 0.8768


In [ ]:
log_model_artifact(
    checkpoint_path=vgg_path,
    artifact_name="vgg16_transfer_learning_best_model"
)


best_epoch,▁
best_val_accuracy,▁
epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁
test_loss,▁
test_macro_f1,▁
test_weighted_f1,▁
train_accuracy,▁▆▇▇██████
train_loss,█▃▂▂▁▂▁▂▁▁
val_accuracy,▁▅████▆▆▆█
+1,...


Model artifact logged: vgg16_transfer_learning_best_model
